# Exploring NHL API to get Data for Expected Goals Model

In [1]:
import pandas as pd
import requests
import json
from typing import List
import time

## Display

In [2]:
def adjust_df_display(dimension, action):
    """This function when called adjusts the output display of pandas dataframes. It either changes the max_columns or max_rows to infinite or resets those
    values to their default display limits.

    Args:
        dimension (string): Display dimension of a dataframe to alter; should be either "columns" or "rows".
        action (string): Action to be carried out on display settings; should be either "max" or "limit".
    """
    if dimension == "columns" and action == "max":
        pd.set_option('display.max_columns', None)
    elif dimension == "rows" and action == "max":
        pd.set_option('display.max_rows', None)
    elif dimension == "columns" and action == "limit":
        pd.reset_option('max_columns')
    else:
        pd.reset_option('max_rows')

In [3]:
adjust_df_display("columns", "max")
adjust_df_display("rows", "max")

## API Data

In [4]:
url = 'https://api-web.nhle.com/v1/gamecenter/2024020954/play-by-play'

# Send GET request to fetch game data
response = requests.get(url)

# Check if reponse was successful
if response.status_code == 200:
    # Parse JSON response
    data = response.json()
    # Pretty print the JSON with an indentation of 4 spaces
    pretty_json = json.dumps(data, indent=4)
    # Output data
    #print(pretty_json)
    
    # Step 2: Extract the plays list
    plays = data.get("plays", [])

    # Step 3: Flatten the data
    # We'll normalize both top-level and nested dictionaries
    #df = pd.json_normalize(plays, sep="_")
    df = pd.json_normalize(data, sep="_")
    
else:
    print(f"Error: {response.status_code}")

In [5]:
print(data.keys())

dict_keys(['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock', 'displayPeriod', 'maxPeriods', 'gameOutcome', 'plays', 'rosterSpots', 'regPeriods', 'summary'])


In [6]:
play_data = data.get("plays", [])
print(f"Total events: {len(play_data)}")

Total events: 336


In [7]:
# Inspect the first few events
for play in play_data[:3]:
    print(json.dumps(play, indent=2))

{
  "eventId": 52,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:00",
  "timeRemaining": "20:00",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 520,
  "typeDescKey": "period-start",
  "sortOrder": 8
}
{
  "eventId": 51,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:00",
  "timeRemaining": "20:00",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 502,
  "typeDescKey": "faceoff",
  "sortOrder": 11,
  "details": {
    "eventOwnerTeamId": 28,
    "losingPlayerId": 8482116,
    "winningPlayerId": 8477505,
    "xCoord": 0,
    "yCoord": 0,
    "zoneCode": "N"
  }
}
{
  "eventId": 103,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:09",
  "timeRemaining": "19:51",
  "situationCode": "1551",
  "homeTeamDefending

In [101]:
#print(json.dumps(data, indent=2))
#data

In [ ]:
print(json.dumps(plays, indent=2))  # Pretty-print entire event

In [10]:
df = pd.json_normalize(plays, sep='_')
df.head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
0,52,00:00,20:00,1551,left,520,period-start,8,1,REG,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,51,00:00,20:00,1551,left,502,faceoff,11,1,REG,3,28.0,8482116.0,8477505.0,0.0,0.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,103,00:09,19:51,1551,left,506,shot-on-goal,12,1,REG,3,28.0,NaN,NaN,-55.0,1.0,O,wrist,8477505.0,8476999.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,00:10,19:50,1551,left,516,stoppage,13,1,REG,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,goalie-stopped-after-sog,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,53,00:10,19:50,1551,left,502,faceoff,15,1,REG,3,9.0,8477505.0,8481596.0,-69.0,-22.0,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Event Type: 'goal'

In [11]:
df[df['typeDescKey'] == 'goal']

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
57,264,11:05,08:55,1541,left,505,goal,171,1,REG,3,9.0,NaN,NaN,73.0,4.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8481596.0,12.0,8482092.0,10.0,0.0,1.0,https://nhl.com/video/sjs-ott-pinto-scores-shg...,https://nhl.com/fr/video/sjs-ott-pinto-marque-...,6.369509e+12,6.369509e+12,6.369509e+12,6.369507e+12,NaN,NaN
125,579,05:17,14:43,1541,right,505,goal,350,2,REG,3,28.0,NaN,NaN,83.0,23.0,O,wrist,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8475726.0,22.0,8484227.0,18.0,1.0,1.0,https://nhl.com/video/sjs-ott-toffoli-scores-g...,NaN,6.369510e+12,NaN,NaN,NaN,8484801.0,25.0
157,641,09:47,10:13,1541,right,505,goal,406,2,REG,3,28.0,NaN,NaN,37.0,4.0,O,slap,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8480043.0,5.0,8477505.0,18.0,2.0,1.0,https://nhl.com/video/sjs-ott-liljegren-scores...,https://nhl.com/fr/video/sjs-ott-liljegren-mar...,6.369510e+12,6.369509e+12,6.369508e+12,6.369510e+12,8480011.0,5.0
228,832,01:26,18:34,1351,left,505,goal,551,3,REG,3,9.0,NaN,NaN,56.0,-16.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8480801.0,22.0,8482116.0,42.0,2.0,2.0,https://nhl.com/video/sjs-ott-tkachuk-scores-p...,NaN,6.369509e+12,NaN,6.369510e+12,6.369509e+12,8482105.0,31.0
234,844,03:00,17:00,1551,left,505,goal,563,3,REG,3,9.0,NaN,NaN,77.0,-13.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8482116.0,19.0,8482092.0,11.0,2.0,3.0,https://nhl.com/video/sjs-ott-stutzle-scores-g...,https://nhl.com/fr/video/sjs-ott-stutzle-marqu...,6.369512e+12,6.369510e+12,6.369510e+12,6.369510e+12,NaN,NaN
278,937,08:31,11:29,1551,left,505,goal,661,3,REG,3,9.0,NaN,NaN,84.0,7.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8474102.0,2.0,8478469.0,24.0,2.0,4.0,https://nhl.com/video/sjs-ott-perron-scores-go...,https://nhl.com/fr/video/sjs-ott-perron-marque...,6.369512e+12,6.369510e+12,6.369513e+12,6.369512e+12,8480208.0,31.0
327,1071,18:33,01:27,0641,left,505,goal,802,3,REG,3,28.0,NaN,NaN,-80.0,-7.0,O,wrist,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8484227.0,10.0,8482667.0,31.0,3.0,4.0,https://nhl.com/video/sjs-ott-smith-scores-ppg...,https://nhl.com/fr/video/will-smith-with-a-pow...,6.369511e+12,6.369511e+12,6.369512e+12,6.369511e+12,8484801.0,26.0
329,1078,19:00,01:00,0651,left,505,goal,808,3,REG,3,9.0,NaN,NaN,20.0,-29.0,N,wrist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8478020.0,6.0,8482245.0,7.0,3.0,5.0,https://nhl.com/video/sjs-ott-amadio-scores-em...,https://nhl.com/fr/video/sjs-ott-amadio-ott-ma...,6.

In [12]:
df[df['typeDescKey'] == 'shot-on-goal'].head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
2,103,00:09,19:51,1551,left,506,shot-on-goal,12,1,REG,3,28.0,NaN,NaN,-55.0,1.0,O,wrist,8477505.0,8476999.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,131,02:11,17:49,1551,left,506,shot-on-goal,40,1,REG,3,28.0,NaN,NaN,-56.0,11.0,O,wrist,8480848.0,8476999.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,140,02:57,17:03,1551,left,506,shot-on-goal,52,1,REG,3,28.0,NaN,NaN,-62.0,-15.0,O,wrist,8484911.0,8476999.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,148,03:34,16:26,1551,left,506,shot-on-goal,61,1,REG,3,9.0,NaN,NaN,51.0,-18.0,O,wrist,8480801.0,8477970.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,155,03:59,16:01,1551,left,506,shot-on-goal,67,1,REG,3,28.0,NaN,NaN,-54.0,32.0,O,wrist,8482144.0,8476999.0,4.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df[df['typeDescKey'] == 'missed-shot'].head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
6,115,01:09,18:51,1551,left,507,missed-shot,26,1,REG,3,9.0,NaN,NaN,41.0,37.0,O,wrist,8480801.0,8477970.0,NaN,NaN,high-and-wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,116,01:12,18:48,1551,left,507,missed-shot,28,1,REG,3,9.0,NaN,NaN,54.0,-39.0,O,wrist,8482245.0,8477970.0,NaN,NaN,wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,186,06:06,13:54,1551,left,507,missed-shot,98,1,REG,3,28.0,NaN,NaN,-50.0,-2.0,O,wrist,8477505.0,8476999.0,NaN,NaN,wide-right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,57,07:26,12:34,1551,left,507,missed-shot,113,1,REG,3,28.0,NaN,NaN,-71.0,-11.0,O,tip-in,8475726.0,8476999.0,NaN,NaN,wide-right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,208,07:41,12:19,1551,left,507,missed-shot,122,1,REG,3,28.0,NaN,NaN,-85.0,8.0,O,backhand,8479316.0,8476999.0,NaN,NaN,wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Old version
def get_new_shot_events(game_id: str) -> List[dict]:
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {game_id}")
        return []

    data = response.json()
    plays = data.get('plays', [])
    
    shot_events = []
    for play in plays:
        event_type = play.get('typeDescKey')
        if event_type in ['shot-on-goal', 'missed-shot', 'goal']:
            coordinates = play.get('coordinates', {})
            players = play.get('details', {}).get('players', [])
            
            shooter = next((p['player']['fullName'] for p in players if p['playerType'] in ['Shooter', 'Scorer']), None)
            goalie = next((p['player']['fullName'] for p in players if p['playerType'] == 'Goalie'), None)

            shot_events.append({
                'game_id': game_id,
                'event_type': event_type,
                'period': play.get('period'),
                'period_time': play.get('timeInPeriod'),
                'team': play.get('team', {}).get('name'),
                'x': coordinates.get('x'),
                'y': coordinates.get('y'),
                'shooter': shooter,
                'goalie': goalie
            })
    
    return shot_events

In [19]:
# Example usage
game_ids = ["2024020954"]
all_shots = []
for gid in game_ids:
    all_shots.extend(get_new_shot_events(gid))

shots_df = pd.DataFrame(all_shots)
shots_df
#print(shots_df.head())

,game_id,event_type,period,period_time,team,x,y,shooter,goalie
0,2024020954,shot-on-goal,None,00:09,None,None,None,None,None
1,2024020954,missed-shot,None,01:09,None,None,None,None,None
2,2024020954,missed-shot,None,01:12,None,None,None,None,None
3,2024020954,shot-on-goal,None,02:11,None,None,None,None,None
4,2024020954,shot-on-goal,None,02:57,None,None,None,None,None
5,2024020954,shot-on-goal,None,03:34,None,None,None,None,None
6,2024020954,shot-on-goal,None,03:59,None,None,None,None,None
7,2024020954,shot-on-goal,None,04:29,None,None,None,None,None
8,2024020954,missed-shot,None,06:06,None,None,None,None,None
9,2024020954,shot-on-goal,None,07:16,None,None,None,None,None


In [20]:
game_id = "2024020954"
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
response = requests.get(url)
data = response.json()

# # Filter for a single shot event
# for play in data.get("plays", []):
#     if play.get("typeDescKey") in ["shot-on-goal", "goal", "missed-shot", "blocked-shot"]:
#         print(json.dumps(play, indent=2))  # Pretty print the full structure
#         break  # Stop after first match
count = 0
for play in data.get("plays", []):
    print(play.keys())
    if play.get("typeDescKey") in ["shot-on-goal", "goal", "missed-shot"]:
        print(json.dumps(play, indent=2))  # Pretty-print entire event
        print("=" * 80)  # Visual separator
        count += 1
        if count == 1:
            break

dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder'])
dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details'])
dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details'])
{
  "eventId": 103,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:09",
  "timeRemaining": "19:51",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 506,
  "typeDescKey": "shot-on-goal",
  "sortOrder": 12,
  "details": {
    "xCoord": -55,
    "yCoord": 1,
    "zoneCode": "O",
    "shotType": "wrist",
    "shootingPlayerId": 8477505,
    "goalieInNetId": 8476999,
    "eventOwnerTeamId": 28,
    "aw

In [21]:
# Old version
def extract_shot_events(play_data: list) -> list[dict]:
    """
    Extracts relevant information from shot-related events (goal, shot-on-goal, missed-shot).
    Returns a list of dictionaries with a consistent schema.
    """
    shot_events = []
    valid_types = {"goal", "shot-on-goal", "missed-shot"}

    for event in play_data:
        event_type = event.get("typeDescKey")
        if event_type not in valid_types:
            continue
        
        details = event.get("details", {})
        shooter_id = (
            details.get("scoringPlayerId") if event_type == "goal"
            else details.get("shootingPlayerId")
        )
        
        try:
            shot_info = {
                "event_type": event_type,
                "period": event.get("periodDescriptor", {}).get("number"),
                "time": event.get("timeInPeriod"),
                "x_coord": details.get("xCoord"),
                "y_coord": details.get("yCoord"),
                "zone": details.get("zoneCode"),
                "shot_type": details.get("shotType"),
                "shooter_id": shooter_id,
                "goalie_id": details.get("goalieInNetId"),
                "team_id": details.get("eventOwnerTeamId"),
                "is_goal": 1 if event_type == "goal" else 0
            }
            shot_events.append(shot_info)
        except Exception as e:
            print(f"Skipping event due to error: {e}")
            continue

    return shot_events

In [22]:
def normalize_coordinates(x: int, y: int, is_home_team: bool, home_team_defending_side: str) -> tuple[int, int]:
    """
    Flip coordinates if the shooting team is attacking left.
    Attacking right is the standard direction.
    """
    if home_team_defending_side is None:
        # TODO: Implement fallback for seasons without 'homeTeamDefendingSide'
        return x, y
    
    if home_team_defending_side == "left":
        attacking_left = not is_home_team
    elif home_team_defending_side == "right":
        attacking_left = is_home_team
    else:
        return x, y  # fallback for unexpected or missing value

    if attacking_left:
        # Flip only x-axis
        return -x, y
    else:
        return x, y

In [23]:
# Most up to date version
def extract_shot_events(data: dict, game_id: str) -> list[dict]:
    """
    Extracts relevant information from shot-related events (goal, shot-on-goal, missed-shot),
    including shooter team info and game context.

    Parameters:
    - data: full dictionary returned from the NHL play-by-play API for one game
    - game_id: string identifier for the game

    Returns:
    - List of dictionaries with structured shot event info.
    """
    shot_events = []
    valid_types = {"goal", "shot-on-goal", "missed-shot"}
    play_data = data.get("plays", [])

    # Team context
    home_team = data.get("homeTeam", {})
    away_team = data.get("awayTeam", {})

    for event in play_data:
        event_type = event.get("typeDescKey")
        if event_type not in valid_types:
            continue

        details = event.get("details", {})
        # Identify shooter ID and their team
        shooter_id = (
            details.get("scoringPlayerId") if event_type == "goal"
            else details.get("shootingPlayerId")
        )
        shooter_team_id = details.get("eventOwnerTeamId")

        # Determine if shooter is on home or away team
        is_home_team = shooter_team_id == home_team.get("id")
        shooter_team_abbrev = home_team.get("abbrev") if is_home_team else away_team.get("abbrev")
        opponent_team_abbrev = away_team.get("abbrev") if is_home_team else home_team.get("abbrev")
        
        # Parse situation code and compute strength state
        # 4-digit situation code has this format: A-G-S-H  → Away goalie, Away skaters, Home skaters, Home goalie
        situation_code = event.get("situationCode")
        if isinstance(situation_code, int):
            situation_code = str(situation_code)

        if not (isinstance(situation_code, str) and len(situation_code) == 4 and situation_code.isdigit()):
            strength_state = None
            away_goalie_pulled = None
            home_goalie_pulled = None
            shooting_team_strength_state = None
            shooting_team_strength_diff = None
        else:
            away_goalie_pulled = situation_code[0] == "0"
            home_goalie_pulled = situation_code[3] == "0"
            # strength_state is 'Away skaters'v'Home skaters' e.g. '4v5' for Home team on powerplay
            strength_state = f"{situation_code[1]}v{situation_code[2]}"
            
            # Derive skater counts
            away_skaters = int(situation_code[1])
            home_skaters = int(situation_code[2])

            if is_home_team is True:
                shooting_team_skaters = home_skaters
                defending_team_skaters = away_skaters
            elif is_home_team is False:
                shooting_team_skaters = away_skaters
                defending_team_skaters = home_skaters
            else:
                shooting_team_skaters = None
                defending_team_skaters = None

            if shooting_team_skaters is not None and defending_team_skaters is not None:
                shooting_team_strength_state = f"{shooting_team_skaters}v{defending_team_skaters}"
                shooting_team_strength_diff = shooting_team_skaters - defending_team_skaters
            else:
                shooting_team_strength_state = None
                shooting_team_strength_diff = None
                
        x_coord = details.get("xCoord")
        y_coord = details.get("yCoord")
        home_team_defending_side = event.get("homeTeamDefendingSide")
        x_norm, y_norm = normalize_coordinates(x_coord, y_coord, is_home_team, home_team_defending_side)

        try:
            shot_info = {
                # Game context
                "game_id": game_id,
                "event_id": event.get("eventId"),
                "sort_order": event.get("sortOrder"),
                "period": event.get("periodDescriptor", {}).get("number"),
                "period_type": event.get("periodDescriptor", {}).get("periodType"),
                "time_in_period": event.get("timeInPeriod"),
                "time_remaining": event.get("timeRemaining"),
                "situation_code": situation_code,
                "strength_state": strength_state,
                "away_goalie_pulled": away_goalie_pulled,
                "home_goalie_pulled": home_goalie_pulled,
                "shooting_team_strength_state": shooting_team_strength_state,
                "shooting_team_strength_diff": shooting_team_strength_diff,
                # TODO: Implement logic to compute score_state and score_differential
                "score_state": None,
                "score_differential": None,

                # Player & team info
                "shooter_id": shooter_id,
                "goalie_id": details.get("goalieInNetId"),
                "team_id": shooter_team_id,
                "shooter_team_abbrev": shooter_team_abbrev,
                "opponent_team_abbrev": opponent_team_abbrev,
                "is_home_team": is_home_team,

                # Shot event info
                "event_type": event_type,
                "x_coord_raw": x_coord,
                "y_coord_raw": y_coord,
                "x_coord": x_norm,
                "y_coord": y_norm,
                "zone": details.get("zoneCode"),
                "shot_type": details.get("shotType"),
                "is_goal": 1 if event_type == "goal" else 0,
                # TODO: Implement is_rebound using prior event logic
                "is_rebound": None,
                # TODO: Implement is_rush_shot using transition speed heuristics
                "is_rush_shot": None
            }

            if event_type == "missed-shot":
                shot_info["miss_reason"] = details.get("reason")

            shot_events.append(shot_info)

        except Exception as e:
            print(f"Skipping event due to error: {e}")
            continue

    return shot_events



In [24]:
def get_shot_data_for_game(game_id: str) -> list[dict]:
    try:
        url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return extract_shot_events(data, game_id)
    except Exception as e:
        print(f"Failed for game {game_id}: {e}")
        return []

In [25]:
def get_nhl_team_abbreviations() -> list[str]:
    url = "https://api-web.nhle.com/v1/standings/now"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    abbrevs = []
    for team_record in data.get("standings", []):
        team_abbrev_info = team_record.get("teamAbbrev", {})
        abbrev = team_abbrev_info.get("default")
        if abbrev:
            abbrevs.append(abbrev)

    return sorted(set(abbrevs))

# Example usage
team_abbrevs = get_nhl_team_abbreviations()

In [26]:
def get_all_regular_season_game_ids(season: str, team_abbrevs: list[str]):
    game_ids = set()
    for team in team_abbrevs:
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{team}/{season}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        for g in data.get("games", []):
            if g.get("gameType") == 2:
                game_ids.add(g["id"])
    return sorted(game_ids)

# Example usage
season = '20232024'
game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
print(f"Found {len(game_ids)} regular-season games for {season}")

Found 1312 regular-season games for 20232024


In [27]:
for season in ['20192020', '20202021', '20212022', '20222023', '20232024', '20242025']:
    game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
    print(f"{season} season: {len(game_ids)} regular season games")

20192020 season: 1082 regular season games
20202021 season: 868 regular season games
20212022 season: 1312 regular season games
20222023 season: 1312 regular season games
20232024 season: 1312 regular season games
20242025 season: 1312 regular season games


In [156]:
all_shots = []

seasons = ['20192020', '20202021', '20212022', '20222023', '20232024', '20242025']

for season in seasons:
    print(f"Starting season {season}...")
    game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
    for i, game_id in enumerate(game_ids):
        shots = get_shot_data_for_game(game_id)
        if shots:
            all_shots.extend(shots)
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1} games from season {season}")
        time.sleep(0.1)  # Gentle pacing to avoid hammering server

# Convert to DataFrame and save
df = pd.DataFrame(all_shots)
#df.to_csv("nhl_shots_2019_2024.csv", index=False)
print("Finished collecting shot data.")
df.shape

Starting season 20192020...
  Processed 50 games from season 20192020
  Processed 100 games from season 20192020
  Processed 150 games from season 20192020
  Processed 200 games from season 20192020
  Processed 250 games from season 20192020
  Processed 300 games from season 20192020
  Processed 350 games from season 20192020
  Processed 400 games from season 20192020
  Processed 450 games from season 20192020
  Processed 500 games from season 20192020
  Processed 550 games from season 20192020
  Processed 600 games from season 20192020
  Processed 650 games from season 20192020
  Processed 700 games from season 20192020
  Processed 750 games from season 20192020
  Processed 800 games from season 20192020
  Processed 850 games from season 20192020
  Processed 900 games from season 20192020
  Processed 950 games from season 20192020
  Processed 1000 games from season 20192020
  Processed 1050 games from season 20192020
Starting season 20202021...
  Processed 50 games from season 2020202

(622481, 32)

In [161]:
df.to_csv("nhl_shots_2019_2024.csv", index=False)

In [160]:
df.shape

(622481, 32)

In [158]:
df.head(10)

,game_id,event_id,sort_order,period,period_type,time_in_period,time_remaining,situation_code,strength_state,away_goalie_pulled,home_goalie_pulled,shooting_team_strength_state,shooting_team_strength_diff,score_state,score_differential,shooter_id,goalie_id,team_id,shooter_team_abbrev,opponent_team_abbrev,is_home_team,event_type,x_coord_raw,y_coord_raw,x_coord,y_coord,zone,shot_type,is_goal,is_rebound,is_rush_shot,miss_reason
0,2019020001,10,11,1,REG,00:25,19:35,1551,5v5,False,False,5v5,0,None,None,8480801,8475883.0,9,OTT,TOR,False,goal,85.0,-1.0,85.0,-1.0,O,tip-in,1,None,None,NaN
1,2019020001,12,15,1,REG,00:38,19:22,1551,5v5,False,False,5v5,0,None,None,8479458,8475883.0,9,OTT,TOR,False,missed-shot,28.0,-37.0,28.0,-37.0,O,slap,0,None,None,wide-of-net
2,2019020001,15,28,1,REG,01:31,18:29,1451,4v5,False,False,5v4,1,None,None,8476853,8467950.0,10,TOR,OTT,True,shot-on-goal,-32.0,-2.0,32.0,-2.0,O,snap,0,None,None,NaN
3,2019020001,18,33,1,REG,01:58,18:02,1451,4v5,False,False,5v4,1,None,None,8478483,8467950.0,10,TOR,OTT,True,missed-shot,-46.0,-16.0,46.0,-16.0,O,wrist,0,None,None,wide-of-net
4,2019020001,19,44,1,REG,03:09,16:51,1551,5v5,False,False,5v5,0,None,None,8478857,8467950.0,10,TOR,OTT,True,missed-shot,-64.0,-4.0,64.0,-4.0,O,tip-in,0,None,None,wide-of-net
5,2019020001,20,45,1,REG,03:23,16:37,1551,5v5,False,False,5v5,0,None,None,8476331,8475883.0,9,OTT,TOR,False,shot-on-goal,63.0,-6.0,63.0,-6.0,O,snap,0,None,None,NaN
6,2019020001,21,53,1,REG,03:56,16:04,1551,5v5,False,False,5v5,0,None,None,8476853,8467950.0,10,TOR,OTT,True,shot-on-goal,-59.0,-20.0,59.0,-20.0,O,wrist,0,None,None,NaN
7,2019020001,23,61,1,REG,04:41,15:19,1551,5v5,False,False,5v5,0,None,None,8479675,8467950.0,10,TOR,OTT,True,missed-shot,-86.0,4.0,86.0,4.0,O,tip-in,0,None,None,over-net
8,2019020001,24,62,1,REG,04:47,15:13,1551,5v5,False,False,5v5,0,None,None,8475197,8467950.0,10,TOR,OTT,True,shot-on-goal,-42.0,-29.0,42.0,-29.0,O,slap,0,None,None,NaN
9,2019020001,25,63,1,REG,04:53,15:07,1551,5v5,False,False,5v5,0,None,None,8475197,8467950.0,10,TOR,OTT,True,shot-on-goal,-52.0,-7.0,52.0,-7.0,O,slap,0,None,None,NaN


In [159]:
df.tail(10)

,game_id,event_id,sort_order,period,period_type,time_in_period,time_remaining,situation_code,strength_state,away_goalie_pulled,home_goalie_pulled,shooting_team_strength_state,shooting_team_strength_diff,score_state,score_differential,shooter_id,goalie_id,team_id,shooter_team_abbrev,opponent_team_abbrev,is_home_team,event_type,x_coord_raw,y_coord_raw,x_coord,y_coord,zone,shot_type,is_goal,is_rebound,is_rush_shot,miss_reason
622471,2024021312,1043,689,3,REG,15:33,04:27,1551,5v5,False,False,5v5,0,None,None,8480865,8482982.0,2,NYI,CBJ,False,shot-on-goal,-39.0,-24.0,39.0,-24.0,O,slap,0,None,None,NaN
622472,2024021312,1050,697,3,REG,16:04,03:56,1551,5v5,False,False,5v5,0,None,None,8483553,8482982.0,2,NYI,CBJ,False,shot-on-goal,-69.0,16.0,69.0,16.0,O,wrist,0,None,None,NaN
622473,2024021312,1061,709,3,REG,17:14,02:46,1551,5v5,False,False,5v5,0,None,None,8480865,8482982.0,2,NYI,CBJ,False,shot-on-goal,-82.0,-12.0,82.0,-12.0,O,backhand,0,None,None,NaN
622474,2024021312,1065,714,3,REG,17:24,02:36,1551,5v5,False,False,5v5,0,None,None,8476432,8477405.0,29,CBJ,NYI,True,shot-on-goal,69.0,20.0,69.0,20.0,O,snap,0,None,None,NaN
622475,2024021312,1072,720,3,REG,17:52,02:08,1551,5v5,False,False,5v5,0,None,None,8476422,8482982.0,2,NYI,CBJ,False,shot-on-goal,-78.0,-10.0,78.0,-10.0,O,tip-in,0,None,None,NaN
622476,2024021312,1074,721,3,REG,17:54,02:06,1551,5v5,False,False,5v5,0,None,None,8475231,8482982.0,2,NYI,CBJ,False,shot-on-goal,-80.0,-4.0,80.0,-4.0,O,backhand,0,None,None,NaN
622477,2024021312,1078,727,3,REG,18:27,01:33,1551,5v5,False,False,5v5,0,None,None,8478460,8477405.0,29,CBJ,NYI,True,shot-on-goal,31.0,33.0,31.0,33.0,O,slap,0,None,None,NaN
622478,2024021312,1082,732,3,REG,18:37,01:23,1551,5v5,False,False,5v5,0,None,None,8484166,8477405.0,29,CBJ,NYI,True,goal,62.0,32.0,62.0,32.0,O,wrist,1,None,None,NaN
622479,2024021312,1086,736,3,REG,19:20,00:40,1551,5v5,False,False,5v5,0,None,None,8483485,8477405.0,29,CBJ,NYI,True,shot-on-goal,37.0,31.0,37.0,31.0,O,slap,0,None,None,NaN
622480,2024021312,87,740,3,REG,19:27,00:33,1551,5v5,False,False,5v5,0,None,None,8474709,8482982.0,2,NYI,CBJ,False,missed-shot,-49.0,-22.0,49.0,-22.0,O,slap,0,None,None,wide-left


In [123]:
game_ids = ["2024020954", "2024020955", "2024020956", "2024020957", "2024020958", "2024020959", "2024020960", "2024020961", "2024020962", "2024020963"]

In [124]:
all_shots = []

for game_id in game_ids:
    shots = get_shot_data_for_game(game_id)
    all_shots.extend(shots)

df = pd.DataFrame(all_shots)
df.head(20)

,game_id,event_id,sort_order,period,period_type,time_in_period,time_remaining,situation_code,strength_state,away_goalie_pulled,home_goalie_pulled,shooting_team_strength_state,shooting_team_strength_diff,score_state,score_differential,shooter_id,goalie_id,team_id,shooter_team_abbrev,opponent_team_abbrev,is_home_team,event_type,x_coord_raw,y_coord_raw,x_coord,y_coord,zone,shot_type,is_goal,is_rebound,is_rush_shot,miss_reason
0,2024020954,103,12,1,REG,00:09,19:51,1551,5v5,False,False,5v5,0,None,None,8477505,8476999.0,28,SJS,OTT,False,shot-on-goal,-55,1,55,1,O,wrist,0,None,None,NaN
1,2024020954,115,26,1,REG,01:09,18:51,1551,5v5,False,False,5v5,0,None,None,8480801,8477970.0,9,OTT,SJS,True,missed-shot,41,37,41,37,O,wrist,0,None,None,high-and-wide-left
2,2024020954,116,28,1,REG,01:12,18:48,1551,5v5,False,False,5v5,0,None,None,8482245,8477970.0,9,OTT,SJS,True,missed-shot,54,-39,54,-39,O,wrist,0,None,None,wide-left
3,2024020954,131,40,1,REG,02:11,17:49,1551,5v5,False,False,5v5,0,None,None,8480848,8476999.0,28,SJS,OTT,False,shot-on-goal,-56,11,56,11,O,wrist,0,None,None,NaN
4,2024020954,140,52,1,REG,02:57,17:03,1551,5v5,False,False,5v5,0,None,None,8484911,8476999.0,28,SJS,OTT,False,shot-on-goal,-62,-15,62,-15,O,wrist,0,None,None,NaN
5,2024020954,148,61,1,REG,03:34,16:26,1551,5v5,False,False,5v5,0,None,None,8480801,8477970.0,9,OTT,SJS,True,shot-on-goal,51,-18,51,-18,O,wrist,0,None,None,NaN
6,2024020954,155,67,1,REG,03:59,16:01,1551,5v5,False,False,5v5,0,None,None,8482144,8476999.0,28,SJS,OTT,False,shot-on-goal,-54,32,54,32,O,wrist,0,None,None,NaN
7,2024020954,162,73,1,REG,04:29,15:31,1551,5v5,False,False,5v5,0,None,None,8482144,8476999.0,28,SJS,OTT,False,shot-on-goal,46,39,-46,39,D,wrist,0,None,None,NaN
8,2024020954,186,98,1,REG,06:06,13:54,1551,5v5,False,False,5v5,0,None,None,8477505,8476999.0,28,SJS,OTT,False,missed-shot,-50,-2,50,-2,O,wrist,0,None,None,wide-right
9,2024020954,199,112,1,REG,07:16,12:44,1551,5v5,False,False,5v5,0,None,None,8478013,8476999.0,28,SJS,OTT,False,shot-on-goal,-36,25,36,25,O,slap,0,None,None,NaN


In [ ]:
def audit_event_fields(game_id: str):
    data = get_shot_data_for_game(game_id)
    if not data:
        print(f"No data found for {game_id}")
        return

    sample_event = data[0]
    print(f"Fields in event {sample_event['event_id']} from {game_id}:")
    for key in sample_event.keys():
        print(f"- {key}")

In [128]:
def audit_event_fields(game_id: str):
    try:
        url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return f"Failed to retrieve data for game {game_id}: {e}"

    play_data = data.get("plays", [])
    valid_types = {"shot-on-goal", "missed-shot"}

    for event in play_data:
        if event.get("typeDescKey") in valid_types:
            details = event.get("details", {})
            return {
                "game_id": game_id,
                "event_id": event.get("eventId"),
                "top_level_keys": list(event.keys()),
                "details_keys": list(details.keys()) if isinstance(details, dict) else "No details"
            }

    return f"No valid shot events found in game {game_id}"

game_ids = [
    "2015020100", "2016020100", "2017020100", "2018020100", "2019020100", "2020020100",
    "2021020100", "2022020100", "2023020100", "2024020100"
]

for gid in game_ids:
    result = audit_event_fields(gid)
    print(result)

{'game_id': '2015020100', 'event_id': 54, 'top_level_keys': ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'typeCode', 'typeDescKey', 'sortOrder', 'details'], 'details_keys': ['xCoord', 'yCoord', 'zoneCode', 'shotType', 'shootingPlayerId', 'goalieInNetId', 'eventOwnerTeamId', 'awaySOG', 'homeSOG']}
{'game_id': '2016020100', 'event_id': 57, 'top_level_keys': ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'typeCode', 'typeDescKey', 'sortOrder', 'details'], 'details_keys': ['xCoord', 'yCoord', 'zoneCode', 'shotType', 'shootingPlayerId', 'goalieInNetId', 'eventOwnerTeamId', 'awaySOG', 'homeSOG']}
{'game_id': '2017020100', 'event_id': 9, 'top_level_keys': ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'typeCode', 'typeDescKey', 'sortOrder', 'details'], 'details_keys': ['xCoord', 'yCoord', 'zoneCode', 'shotType', 'shootingPlayerId', 'goalieInNetId', 'eventOwnerTeamId', 'awaySOG', 'homeSOG

In [127]:
def print_first_shot_event(game_id: str):
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    response = requests.get(url)
    data = response.json()

    print(f"\n==== Game ID: {game_id} ====")
    plays = data.get("plays", [])
    for play in plays:
        if play.get("typeDescKey") in {"goal", "shot-on-goal", "missed-shot"}:
            print("Full event:")
            print(json.dumps(play, indent=2))
            print("\nDetails field only:")
            print(json.dumps(play.get("details", {}), indent=2))
            return
    print("No shot events found.")

# Sample Game IDs from 2017–18 and 2018–19 seasons
game_ids = [
    "2017020100", "2018020100"  # 2018–19 season
]

for gid in game_ids:
    print_first_shot_event(gid)


==== Game ID: 2017020100 ====
Full event:
{
  "eventId": 9,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:23",
  "timeRemaining": "19:37",
  "situationCode": "1551",
  "typeCode": 506,
  "typeDescKey": "shot-on-goal",
  "sortOrder": 14,
  "details": {
    "xCoord": -1,
    "yCoord": -36,
    "zoneCode": "N",
    "shotType": "wrist",
    "shootingPlayerId": 8476457,
    "goalieInNetId": 8476341,
    "eventOwnerTeamId": 22,
    "awaySOG": 1,
    "homeSOG": 0
  }
}

Details field only:
{
  "xCoord": -1,
  "yCoord": -36,
  "zoneCode": "N",
  "shotType": "wrist",
  "shootingPlayerId": 8476457,
  "goalieInNetId": 8476341,
  "eventOwnerTeamId": 22,
  "awaySOG": 1,
  "homeSOG": 0
}

==== Game ID: 2018020100 ====
Full event:
{
  "eventId": 53,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:58",
  "timeRemaining": "19:02",
  "situationCode": "1